In [ ]:
import os
import sys
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import shutil
import glob
from PIL import Image

ROOT = os.path.abspath(".")
if ROOT not in sys.path:
    sys.path.append(ROOT)
os.chdir(ROOT)

from config import Config
from model import SingleStreamDiT
from latents import decode_latents_to_image, prepare_latents_for_decode
from samplers import run_sampling_pipeline
from model_loader import load_vae

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
Config.device = DEVICE
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

print(f"Jupyter environment configured on device: {DEVICE}")

In [ ]:
FILENAME = "ema_epoch_296.pt"   
latest_checkpoint = os.path.join(Config.checkpoint_dir, FILENAME)
CH_BASE_NAME = os.path.splitext(FILENAME)[0]

print(f"Loading EMA checkpoint: {os.path.basename(latest_checkpoint)}")

print("Loading SingleStreamDiT backbone...")
model = SingleStreamDiT(
    in_channels=Config.in_channels,
    patch_size=Config.patch_size,
    hidden_size=Config.hidden_size,
    depth=Config.depth,
    num_heads=Config.num_heads,
    text_embed_dim=Config.text_embed_dim,
    refiner_depth=Config.refiner_depth,
).to(DEVICE, Config.dtype)

checkpoint_data = torch.load(latest_checkpoint, map_location=DEVICE)

model.load_state_dict(checkpoint_data)
model.eval()

print("Loading VAE model...")
vae = load_vae()
print("System successfully initialized and loaded EMA weights!")

In [ ]:
target_filename = "10003724.pt"
target_file = os.path.join(Config.cache_dir, target_filename)  

if not os.path.exists(target_file):
    raise FileNotFoundError(f"Target cache file not found at: {target_file}")

print(f"Extracting target conditions from: {target_filename}")

data = torch.load(target_file, map_location=DEVICE)
h, w = data["height"], data["width"]

if "text_embeds_list" in data:
    text_embeds = data["text_embeds_list"][0].unsqueeze(0).to(DEVICE, Config.dtype)
    text_mask = data["attention_mask_list"][0].unsqueeze(0).to(DEVICE)
else:
    text_embeds = data["text_embeds"].unsqueeze(0).to(DEVICE, Config.dtype)
    text_mask = data["attention_mask"].unsqueeze(0).to(DEVICE)

uncond_embeds = torch.zeros_like(text_embeds)
uncond_mask = text_mask.clone()
combined_text_embeds = torch.cat([uncond_embeds, text_embeds], dim=0)
combined_mask = torch.cat([uncond_mask, text_mask], dim=0)

torch_generator = torch.Generator(device=DEVICE).manual_seed(42)
initial_noise = torch.randn(1, Config.in_channels, h // Config.vae_downsample_factor, 
                            w // Config.vae_downsample_factor, generator=torch_generator, 
                            device=DEVICE, dtype=Config.dtype)

print(f"Successfully loaded conditions. Image dimensions: {w}x{h}")

In [ ]:
print("Generating CONTROL image with Conv2D layers ACTIVE...")
with torch.no_grad():
    with torch.amp.autocast("cuda", dtype=Config.dtype):
        latents_control = run_sampling_pipeline(model=model, initial_noise=initial_noise.clone(), 
                                                steps=30, combined_text_embeds=combined_text_embeds, 
                                                cfg=5.0, text_mask=combined_mask, sampler_type="euler",
                                                scheduler_type="uniform", shift_val=2.5)

img_control = decode_latents_to_image(vae, latents_control, DEVICE)
print("Control image successfully generated and held in memory.")

In [ ]:
original_convs = {}
for name, block_list in [("blocks", model.blocks), ("noise_refiner", model.noise_refiner)]:
    for i, block in enumerate(block_list):
        if hasattr(block, 'ffn') and hasattr(block.ffn, 'dwconv'):
            original_convs[(name, i)] = block.ffn.dwconv
            block.ffn.dwconv = nn.Identity()

print("Generating EXPERIMENTAL image with Conv2D layers INACTIVE...")
with torch.no_grad():
    with torch.amp.autocast("cuda", dtype=Config.dtype):
        latents_ablated = run_sampling_pipeline(model=model, initial_noise=initial_noise.clone(), 
                                                steps=30, combined_text_embeds=combined_text_embeds, 
                                                cfg=5.0, text_mask=combined_mask, sampler_type="euler", 
                                                scheduler_type="uniform", shift_val=2.5)

img_ablated = decode_latents_to_image(vae, latents_ablated, DEVICE)

print("Restoring original Conv2D layers...")
for (name, i), conv in original_convs.items():
    getattr(model, name)[i].ffn.dwconv = conv

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
axes[0].imshow(img_control)
axes[0].set_title(f"Control Group (Active)\n[{CH_BASE_NAME}]", fontsize=12)
axes[0].axis("off")

axes[1].imshow(img_ablated)
axes[1].set_title(f"Ablated Group (Inactive)\n[{CH_BASE_NAME}]", fontsize=12)
axes[1].axis("off")

plt.tight_layout()

comparison_plot_filename = f"ablation_comparison_{CH_BASE_NAME}.png"
plt.savefig(comparison_plot_filename, dpi=150, bbox_inches='tight')
print(f"Saved side-by-side comparison plot as: {comparison_plot_filename}")

plt.show()

assert not isinstance(model.blocks[8].ffn.dwconv, nn.Identity), "Restore failed!"
assert not isinstance(model.noise_refiner[0].ffn.dwconv, nn.Identity), "Restore failed!"

In [ ]:
weight_tensor = model.blocks[8].ffn.dwconv.weight.detach().cpu().to(torch.float32)
weight_tensor = weight_tensor.squeeze(1)

dirac_ref = torch.zeros_like(weight_tensor)
center = weight_tensor.shape[-1] // 2
dirac_ref[:, center, center] = 1.0

drift = (weight_tensor - dirac_ref).abs().mean(dim=(1, 2))

top_16_indices = torch.argsort(drift, descending=True)[:16]

fig, axes = plt.subplots(4, 4, figsize=(10, 10))
fig.suptitle(f"Top 16 Learned Conv2D Filter Kernels (Block 8)\n[Sorted by Drift-From-Init | {CH_BASE_NAME}]", fontsize=14, y=0.98)

for i, idx in enumerate(top_16_indices):
    ax = axes[i // 4, i % 4]
    kernel_weights = weight_tensor[idx].numpy()
    
    vmax = max(abs(kernel_weights.min()), abs(kernel_weights.max()))
    im = ax.imshow(kernel_weights, cmap="coolwarm", interpolation="nearest", vmin=-vmax, vmax=vmax)
    
    ax.set_title(f"Channel {idx.item()}\nDrift: {drift[idx].item():.5f}\nVar: {torch.var(weight_tensor[idx]).item():.5f}", fontsize=8)
    ax.axis("off")
    fig.colorbar(im, ax=ax, shrink=0.6)

plt.tight_layout()

weight_plot_filename = f"weights_drift_{CH_BASE_NAME}.png"
plt.savefig(weight_plot_filename, dpi=150, bbox_inches='tight')
print(f"Saved active weight drift visualization as: {weight_plot_filename}")

plt.show()

In [ ]:
total_drift = 0.0
layer_count = 0

for name, block_list in [("blocks", model.blocks), ("noise_refiner", model.noise_refiner)]:
    for i, block in enumerate(block_list):
        if hasattr(block, 'ffn') and hasattr(block.ffn, 'dwconv'):
            w = block.ffn.dwconv.weight.detach().cpu().to(torch.float32).squeeze(1)
            
            ref = torch.zeros_like(w)
            c = w.shape[-1] // 2
            ref[:, c, c] = 1.0
            
            layer_drift = (w - ref).abs().mean().item()
            total_drift += layer_drift
            layer_count += 1

global_average_drift = total_drift / layer_count

print("="*50)
print(f"     GLOBAL CONVERGENCE DIAGNOSTIC: {CH_BASE_NAME}     ")
print("="*50)
print(f"Analyzed Convolutional Layers : {layer_count}")
print(f"Global Average Weight Drift   : {global_average_drift:.8f}")
print("="*50)
print("\n[DIAGNOSTIC CRITERIA]")
print("Note on Weight Drift Metric:")
print(" - Weight drift measures cumulative parameter movement away from Dirac init.")
print(" - Due to AdamW weight decay, drift will monotonically increase over training steps.")
print(" - Do NOT use distance from Dirac init to measure convergence.")
print(" - To measure convergence, evaluate step-over-step weight delta ||W_t - W_{t-1}|| or loss plateaus.")

In [ ]:
intercepted_activations = {}

def hook_fn(module, input_args, output_tensor):
    intercepted_activations['before'] = input_args[0].detach().cpu().to(torch.float32)
    intercepted_activations['after'] = output_tensor.detach().cpu().to(torch.float32)

hook_target = model.blocks[8].ffn.dwconv
handle = hook_target.register_forward_hook(hook_fn)

print("Executing a single forward step to extract activations...")
dummy_t = torch.tensor([0.5], device=DEVICE).to(Config.dtype)
with torch.no_grad():
    _ = model(x=initial_noise.clone(), t=dummy_t, text_embeds=text_embeds, text_mask=text_mask)

handle.remove()

def project_channels_to_rgb(tensor_bchw):
    tensor_chw = tensor_bchw[0]
    C, H, W = tensor_chw.shape
    flat_features = tensor_chw.view(C, -1).t()
    centered_features = flat_features - flat_features.mean(dim=0, keepdim=True)
    U, S, V = torch.linalg.svd(centered_features, full_matrices=False)
    projected = torch.matmul(centered_features, V[:3, :].t())
    min_vals = projected.min(dim=0, keepdim=True).values
    max_vals = projected.max(dim=0, keepdim=True).values
    normalized_proj = (projected - min_vals) / (max_vals - min_vals + 1e-6)
    return normalized_proj.view(H, W, 3).numpy()

if 'before' in intercepted_activations and 'after' in intercepted_activations:
    print("Projecting latent feature spaces using SVD...")
    rgb_before = project_channels_to_rgb(intercepted_activations['before'])
    rgb_after = project_channels_to_rgb(intercepted_activations['after'])
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 7))
    axes[0].imshow(rgb_before)
    axes[0].set_title(f"Latent Features BEFORE Conv2D (Pure MLP/SwiGLU)\n[{CH_BASE_NAME}]", fontsize=12)
    axes[0].axis("off")
    
    axes[1].imshow(rgb_after)
    axes[1].set_title(f"Latent Features AFTER Conv2D (Spatial Filtered)\n[{CH_BASE_NAME}]", fontsize=12)
    axes[1].axis("off")
    
    plt.tight_layout()
    
    svd_plot_filename = f"svd_activations_{CH_BASE_NAME}.png"
    plt.savefig(svd_plot_filename, dpi=150, bbox_inches='tight')
    print(f"Saved activation map visualization as: {svd_plot_filename}")
    
    plt.show()
else:
    print("Error: Hook failed to capture activations.")

In [ ]:
from samplers import get_schedule, euler_step, get_1d_shifted_time
import math

def capture_trajectory_steps(model, initial_noise, steps, combined_text_embeds, cfg=2.5, text_mask=None, shift_val=2.5):
    device = initial_noise.device
    raw_timesteps = get_schedule(Config.validate_scheduler, steps, device)
    timesteps = get_1d_shifted_time(raw_timesteps, shift_val)
    
    x = initial_noise.clone().to(device=DEVICE, dtype=Config.dtype)
    captured_frames = {}
    
    target_steps = [0, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80, 85, 90, 95, 99]
    
    for i in range(steps):
        t = timesteps[i].view(-1)
        t_next = timesteps[i+1].view(-1)
        dt = t_next - t
        
        if i in target_steps:
            pct = int((i / steps) * 100)
            captured_frames[pct] = x.clone()
            
        with torch.no_grad():
            with torch.amp.autocast("cuda", dtype=Config.dtype):
                x = euler_step(model, x, t, dt, combined_text_embeds, cfg, text_mask)
                x = x.to(dtype=Config.dtype)
                
    captured_frames[100] = x.clone()
    return captured_frames

print("Tracing the transport flow trajectory (100 steps)...")
frames = capture_trajectory_steps(
    model, 
    initial_noise, 
    steps=100, 
    combined_text_embeds=combined_text_embeds, 
    cfg=2.5,
    text_mask=combined_mask, 
    shift_val=2.5
)

decoded_frames = []
percentages = sorted(list(frames.keys()))

for pct in percentages:
    print(f"Decoding path at {pct}%...")
    img = decode_latents_to_image(vae, frames[pct], DEVICE)
    decoded_frames.append((pct, img))

num_plots = len(decoded_frames)
cols = 5
rows = math.ceil(num_plots / cols)

fig, axes = plt.subplots(rows, cols, figsize=(16, 4 * rows), squeeze=False)
axes_flat = axes.flatten()

for i, (pct, img) in enumerate(decoded_frames):
    ax = axes_flat[i]
    ax.imshow(img)
    ax.set_title(f"Trajectory: {pct}% of Flow", fontsize=10)
    ax.axis("off")

for j in range(num_plots, len(axes_flat)):
    axes_flat[j].axis("off")

plt.tight_layout()
trajectory_plot_filename = f"flow_trajectory_{CH_BASE_NAME}.png"
plt.savefig(trajectory_plot_filename, dpi=150, bbox_inches='tight')
print(f"\nSaved trajectory static grid as: {trajectory_plot_filename}")
plt.show()

In [ ]:
print("Compiling frames into an animated GIF...")

gif_frames = [img for _, img in decoded_frames]

gif_filename = f"flow_trajectory_animation_{CH_BASE_NAME}.gif"

gif_frames[0].save(
    gif_filename,
    save_all=True,
    append_images=gif_frames[1:],
    duration=100,
    loop=0,
    lossless=True
)

print(f"\nSuccess! Smooth animated trajectory saved to: {gif_filename}")

In [ ]:
file_A = "10003724.pt"
target_file_A = os.path.join(Config.cache_dir, file_A)  

if not os.path.exists(target_file_A):
    raise FileNotFoundError(f"Target cache file not found at: {target_file_A}")

file_B = "10005361.pt"
target_file_B = os.path.join(Config.cache_dir, file_B)  

if not os.path.exists(target_file_B):
    raise FileNotFoundError(f"Target cache file not found at: {target_file_B}")

print(f"Blending Concept A ({os.path.basename(file_A)}) with Concept B ({os.path.basename(file_B)})...")

data_A = torch.load(target_file_A, map_location=DEVICE)
data_B = torch.load(target_file_B, map_location=DEVICE)

h_A, w_A = int(data_A["height"]), int(data_A["width"])
h_B, w_B = int(data_B["height"]), int(data_B["width"])

h, w = h_A, w_A

def get_cond(data):
    if "text_embeds_list" in data:
        return data["text_embeds_list"][0].unsqueeze(0).to(DEVICE, Config.dtype), data["attention_mask_list"][0].unsqueeze(0).to(DEVICE)
    return data["text_embeds"].unsqueeze(0).to(DEVICE, Config.dtype), data["attention_mask"].unsqueeze(0).to(DEVICE)

embed_A, mask_A = get_cond(data_A)
embed_B, mask_B = get_cond(data_B)

max_len = max(embed_A.shape[1], embed_B.shape[1])
def pad_embed(embed, mask, target_len):
    if embed.shape[1] < target_len:
        pad_size = target_len - embed.shape[1]
        embed = torch.cat([embed, torch.zeros(1, pad_size, embed.shape[2], device=DEVICE, dtype=Config.dtype)], dim=1)
        mask = torch.cat([mask, torch.zeros(1, pad_size, device=DEVICE, dtype=torch.bool)], dim=1)
    return embed, mask

embed_A, mask_A = pad_embed(embed_A, mask_A, max_len)
embed_B, mask_B = pad_embed(embed_B, mask_B, max_len)

gen_A = torch.Generator(device=DEVICE).manual_seed(100)
gen_B = torch.Generator(device=DEVICE).manual_seed(200)
noise_A = torch.randn(1, Config.in_channels, h // Config.vae_downsample_factor, w // Config.vae_downsample_factor, generator=gen_A, device=DEVICE, dtype=Config.dtype)
noise_B = torch.randn(1, Config.in_channels, h // Config.vae_downsample_factor, w // Config.vae_downsample_factor, generator=gen_B, device=DEVICE, dtype=Config.dtype)

alphas = [0.0, 0.25, 0.5, 0.75, 1.0]
morph_images = []

import math

for alpha in alphas:
    print(f"Rendering frame for Alpha = {alpha:.2f}...")
    
    blended_raw = (1.0 - alpha) * noise_A + alpha * noise_B
    norm_factor = math.sqrt((1.0 - alpha)**2 + alpha**2)
    blended_noise = blended_raw / (norm_factor if norm_factor > 0 else 1.0)
    
    blended_embed = (1.0 - alpha) * embed_A + alpha * embed_B
    blended_mask = mask_A | mask_B 
    
    uncond_embed = torch.zeros_like(blended_embed)
    uncond_mask = blended_mask.clone()
    comb_embed = torch.cat([uncond_embed, blended_embed], dim=0)
    comb_mask = torch.cat([uncond_mask, blended_mask], dim=0)
    
    with torch.no_grad():
        with torch.amp.autocast("cuda", dtype=Config.dtype):
            latents = run_sampling_pipeline(
                model=model, 
                initial_noise=blended_noise, 
                steps=30, 
                combined_text_embeds=comb_embed, 
                cfg=2.5,
                text_mask=comb_mask,
                sampler_type="euler",
                scheduler_type="uniform",
                shift_val=2.5
            )
            
    img = decode_latents_to_image(vae, latents, DEVICE)
    morph_images.append((alpha, img))

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
fig.suptitle(f"Conceptual Manifold Morphing (Concept A -> Concept B)\n[CFG = 2.5 | {CH_BASE_NAME}]", fontsize=14, y=0.98)

for i, (alpha, img) in enumerate(morph_images):
    axes[i].imshow(img)
    axes[i].set_title(f"Alpha: {alpha:.2f}", fontsize=10)
    axes[i].axis("off")

plt.tight_layout()
morph_plot_filename = f"conceptual_morph_{CH_BASE_NAME}.png"
plt.savefig(morph_plot_filename, dpi=150, bbox_inches='tight')
print(f"Saved conceptual morphing strip as: {morph_plot_filename}")
plt.show()

In [ ]:
import math

cfg_values = [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0]
sweep_images = []

print("Running Classifier-Free Guidance (CFG) sweep...")
for cfg in cfg_values:
    print(f"Generating image with CFG = {cfg:.1f}...")
    
    with torch.no_grad():
        with torch.amp.autocast("cuda", dtype=Config.dtype):
            latents_cfg = run_sampling_pipeline(
                model=model, 
                initial_noise=initial_noise.clone(), 
                steps=30, 
                combined_text_embeds=combined_text_embeds, 
                cfg=cfg, 
                text_mask=combined_mask,
                sampler_type="euler",
                scheduler_type="uniform",
                shift_val=2.5
            )
            
    img = decode_latents_to_image(vae, latents_cfg, DEVICE)
    sweep_images.append((cfg, img))

num_imgs = len(sweep_images)
fig, axes = plt.subplots(1, num_imgs, figsize=(2.2 * num_imgs, 3.2), squeeze=False)
fig.suptitle(f"Classifier-Free Guidance (CFG) Saturation Sweep\n[{CH_BASE_NAME}]", fontsize=14, y=1.08)

axes_flat = axes.flatten()
for i, (cfg, img) in enumerate(sweep_images):
    axes_flat[i].imshow(img)
    axes_flat[i].set_title(f"CFG: {cfg:.1f}", fontsize=10)
    axes_flat[i].axis("off")

plt.tight_layout()
cfg_sweep_filename = f"cfg_sweep_{CH_BASE_NAME}.png"
plt.savefig(cfg_sweep_filename, dpi=150, bbox_inches='tight')
print(f"Saved CFG sweep visualization as: {cfg_sweep_filename}")
plt.show()

In [ ]:
import os
import torch
import matplotlib.pyplot as plt
from PIL import Image
import torchvision.transforms.functional as TF
from text_encoder import TextEncoderWrapper
from latents import normalize_latents, decode_latents_to_image
from samplers import euler_step, get_1d_shifted_time

UNSEEN_IMAGE_PATH = "image.webp"
CUSTOM_PROMPT = "no humans, star (sky), full moon, night sky, scenery, outdoors, purple flower, cloud, landscape, starry sky, grass, mountain, mountainous horizon, nature, plant"
NOISE_STRENGTH = 0.75

if 'text_encoder' not in globals():
    print("Loading Qwen3 Text Encoder into VRAM...")
    text_encoder = TextEncoderWrapper(dtype=Config.dtype, device=DEVICE)

if not os.path.exists(UNSEEN_IMAGE_PATH):
    raise FileNotFoundError(f"Please place an image at '{UNSEEN_IMAGE_PATH}' or change the path in the cell.")

print(f"Loading image: {UNSEEN_IMAGE_PATH}")
raw_img = Image.open(UNSEEN_IMAGE_PATH).convert("RGB")
img_w, img_h = raw_img.size

def get_aligned_dimensions(w, h, target_res=Config.target_resolution, stride=32):
    target_area = target_res * target_res
    aspect = w / h
    calc_w = math.sqrt(target_area * aspect)
    calc_h = math.sqrt(target_area / aspect)
    bw = int(max(stride, round(calc_w / stride) * stride))
    bh = int(max(stride, round(calc_h / stride) * stride))
    return bw, bh

bw, bh = get_aligned_dimensions(img_w, img_h, target_res=Config.target_resolution, stride=32)
print(f"Original Size: {img_w}x{img_h} -> Native Aspect Preserved Size: {bw}x{bh}")

resized_img = raw_img.resize((bw, bh), resample=Image.LANCZOS)

img_tensor = TF.to_tensor(resized_img).unsqueeze(0).to(DEVICE, Config.dtype)
batch_tensor_vae = TF.normalize(img_tensor, [0.5], [0.5])

print("Compressing image into VAE Latent Space...")
with torch.no_grad():
    with torch.amp.autocast("cuda", enabled=False):
        encoded = vae.encode(batch_tensor_vae.float())
        latents_raw = encoded.latent_dist.mode() if hasattr(encoded, "latent_dist") else encoded[0]

clean_latent = normalize_latents(latents_raw.to(Config.dtype))

start_t = 1.0 - NOISE_STRENGTH
print(f"Injecting {NOISE_STRENGTH*100:.0f}% noise...")
torch_generator = torch.Generator(device=DEVICE).manual_seed(42)
pure_noise = torch.randn_like(clean_latent, generator=torch_generator)

partially_noisy_latent = (1.0 - start_t) * pure_noise + start_t * clean_latent

print(f"Encoding custom prompt: '{CUSTOM_PROMPT}'")
with torch.no_grad():
    embed, mask = text_encoder.encode([CUSTOM_PROMPT])

mask_bool = mask.bool().to(DEVICE)
uncond = torch.zeros_like(embed)
uncond_mask = mask_bool.clone()
comb_embed = torch.cat([uncond, embed], dim=0)
comb_mask = torch.cat([uncond_mask, mask_bool], dim=0)

SHIFT_VAL = 2.5
steps = 50

def invert_shifted_time(t, shift_val):
    s = 1.0 / shift_val
    return t / (s * (1.0 - t) + t)

u_start = invert_shifted_time(torch.tensor(start_t, device=DEVICE, dtype=torch.float32), SHIFT_VAL)
u_schedule = torch.linspace(u_start.item(), 1.0, steps + 1, device=DEVICE)
timesteps = get_1d_shifted_time(u_schedule, shift_val=SHIFT_VAL)

x = partially_noisy_latent.clone()

print("Denoising toward custom prompt using CFG = 1.5...")
for i in range(steps):
    t = timesteps[i].view(-1)
    t_next = timesteps[i + 1].view(-1)
    dt = t_next - t

    with torch.no_grad():
        with torch.amp.autocast("cuda", dtype=Config.dtype):
            x = euler_step(model, x, t, dt, comb_embed, cfg=1.5, text_mask=comb_mask)
            x = x.to(dtype=Config.dtype)

print("Decoding results...")
img_noisy_start = decode_latents_to_image(vae, partially_noisy_latent, DEVICE)
img_translated = decode_latents_to_image(vae, x, DEVICE)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(resized_img)
axes[0].set_title(f"Raw Input ({bw}x{bh})", fontsize=12)
axes[0].axis("off")

axes[1].imshow(img_noisy_start)
axes[1].set_title(f"Noised Latent VAE Decode ({NOISE_STRENGTH*100:.0f}% Noise)\n[Un-denoised latent noise]", fontsize=12)
axes[1].axis("off")

axes[2].imshow(img_translated)
axes[2].set_title(f"Anime-ified Output ({bw}x{bh})\n(CFG = 1.5)", fontsize=12)
axes[2].axis("off")

plt.tight_layout()
img_base = os.path.splitext(os.path.basename(UNSEEN_IMAGE_PATH))[0]
universal_sdedit_filename = f"universal_translation_{img_base}.png"
plt.savefig(universal_sdedit_filename, dpi=150, bbox_inches='tight')
print(f"Saved style translation as: {universal_sdedit_filename}")
plt.show()

In [ ]:
import os
import torch
import matplotlib.pyplot as plt
from latents import normalize_latents, decode_latents_to_image
from samplers import euler_step, get_1d_shifted_time

target_file_A = "10003724.pt"
target_file_A = os.path.join(Config.cache_dir, target_file_A)

if not os.path.exists(target_file_A):
    raise FileNotFoundError(f"Target cache file not found at: {target_file_A}")

target_file_B = "5567876.pt"
target_file_B = os.path.join(Config.cache_dir, target_file_B)

if not os.path.exists(target_file_B):
    raise FileNotFoundError(f"Target cache file not found at: {target_file_B}")

data_A = torch.load(target_file_A, map_location=DEVICE)
data_B = torch.load(target_file_B, map_location=DEVICE)

h_A, w_A = int(data_A["height"]), int(data_A["width"])

print(f"Extracting and normalizing structure latent from Concept A ({target_file_A})...")
clean_latent = normalize_latents(data_A["latents"].unsqueeze(0).to(DEVICE, Config.dtype))

noise_strength = 0.95
start_t = 1.0 - noise_strength

print(f"Injecting {noise_strength*100:.0f}% Gaussian noise...")
torch_generator = torch.Generator(device=DEVICE).manual_seed(42)
pure_noise = torch.randn_like(clean_latent, generator=torch_generator)

partially_noisy_latent = (1.0 - start_t) * pure_noise + start_t * clean_latent

def get_cond(data):
    if "text_embeds_list" in data:
        return data["text_embeds_list"][0].unsqueeze(0).to(DEVICE, Config.dtype), data["attention_mask_list"][0].unsqueeze(0).to(DEVICE)
    return data["text_embeds"].unsqueeze(0).to(DEVICE, Config.dtype), data["attention_mask"].unsqueeze(0).to(DEVICE)

embed_B, mask_B = get_cond(data_B)

uncond_B = torch.zeros_like(embed_B)
uncond_mask_B = mask_B.clone()
comb_embed_B = torch.cat([uncond_B, embed_B], dim=0)
comb_mask_B = torch.cat([uncond_mask_B, mask_B], dim=0)

SHIFT_VAL = 2.5
steps = 20

def invert_shifted_time(t, shift_val):
    s = 1.0 / shift_val
    return t / (s * (1.0 - t) + t)

u_start = invert_shifted_time(torch.tensor(start_t, device=DEVICE, dtype=torch.float32), SHIFT_VAL)
u_schedule = torch.linspace(u_start.item(), 1.0, steps + 1, device=DEVICE)
timesteps = get_1d_shifted_time(u_schedule, shift_val=SHIFT_VAL)

x = partially_noisy_latent.clone()

print(f"Denoising toward Concept B ({target_file_B}) using CFG = 2.5...")
for i in range(steps):
    t = timesteps[i].view(-1)
    t_next = timesteps[i + 1].view(-1)
    dt = t_next - t

    with torch.no_grad():
        with torch.amp.autocast("cuda", dtype=Config.dtype):
            x = euler_step(model, x, t, dt, comb_embed_B, cfg=2.5, text_mask=comb_mask_B)
            x = x.to(dtype=Config.dtype)

print("Decoding latents to image space...")
img_control_A = decode_latents_to_image(vae, clean_latent, DEVICE)
img_noisy_start = decode_latents_to_image(vae, partially_noisy_latent, DEVICE)
img_translated = decode_latents_to_image(vae, x, DEVICE)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(img_control_A)
axes[0].set_title(f"Original Concept A Structure\n[{os.path.basename(target_file_A)}]", fontsize=12)
axes[0].axis("off")

axes[1].imshow(img_noisy_start)
axes[1].set_title(f"Raw Noisy Latent VAE Decode ({NOISE_STRENGTH*100:.0f}% Noise)\n[Expected high-frequency noise output]", fontsize=12)
axes[1].axis("off")

axes[2].imshow(img_translated)
axes[2].set_title(f"SDEdit Style-Translated Output\n(Prompted to {os.path.basename(target_file_B)} | CFG = 2.5)", fontsize=12)
axes[2].axis("off")

plt.tight_layout()
CH_BASE_NAME = f"{os.path.splitext(os.path.basename(target_file_A))[0]}_to_{os.path.splitext(os.path.basename(target_file_B))[0]}"
sdedit_filename = f"sdedit_translation_{CH_BASE_NAME}.png"
plt.savefig(sdedit_filename, dpi=150, bbox_inches='tight')
print(f"Saved SDEdit style-translation as: {sdedit_filename}")
plt.show()

In [ ]:
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from config import Config

USE_EMA = True 

checkpoint_epochs = [92, 172, 296]

ckpt_paths = {}
for ep in checkpoint_epochs:
    possible_paths = [
        os.path.join(Config.checkpoint_dir, f"full_state_epoch_{ep}.pt"),
        os.path.join(Config.checkpoint_dir, f"full_state_final_epoch_{ep}.pt"),
        os.path.join(Config.checkpoint_dir, f"ema_epoch_{ep}.pt")
    ]
    found = None
    for p in possible_paths:
        if os.path.exists(p):
            found = p
            break
    if found is None:
        raise FileNotFoundError(f"Could not find any checkpoint file for epoch {ep} in {Config.checkpoint_dir}")
    ckpt_paths[ep] = found

print(f"Loading checkpoints (Evaluating {'EMA' if USE_EMA else 'RAW'} weights)...")
states = {}
for ep, path in ckpt_paths.items():
    print(f" -> Loading {os.path.basename(path)}")
    data = torch.load(path, map_location="cpu")
    
    if isinstance(data, dict) and ('ema_state_dict' in data or 'model_state_dict' in data):
        dict_key = 'ema_state_dict' if (USE_EMA and 'ema_state_dict' in data) else 'model_state_dict'
        states[ep] = data[dict_key]
    else:
        states[ep] = data

sample_keys = list(states[92].keys())
dwconv_keys = [k for k in sample_keys if "dwconv.weight" in k]

print(f"\nFound {len(dwconv_keys)} Conv2D depthwise filter layers across DiT blocks.\n")

def analyze_interval(ep_start, ep_end):
    delta_epochs = ep_end - ep_start
    s1, s2 = states[ep_start], states[ep_end]
    
    layer_velocities = []
    layer_names = []
    
    total_sq_diff = 0.0
    total_sq_norm = 0.0
    
    for k in dwconv_keys:
        w1 = s1[k].float()
        w2 = s2[k].float()
        
        diff_norm = torch.norm(w2 - w1, p=2).item()
        base_norm = torch.norm(w1, p=2).item()
        
        rel_change_pct = (diff_norm / base_norm) * 100.0 if base_norm > 0 else 0.0
        velocity = rel_change_pct / delta_epochs  # % change per epoch
        
        layer_velocities.append(velocity)
        short_name = k.replace(".ffn.dwconv.weight", "").replace("module.", "")
        layer_names.append(short_name)
        
        total_sq_diff += diff_norm ** 2
        total_sq_norm += base_norm ** 2
        
    global_rel_change = (np.sqrt(total_sq_diff) / np.sqrt(total_sq_norm)) * 100.0
    global_velocity = global_rel_change / delta_epochs
    
    return {
        "velocities": layer_velocities,
        "names": layer_names,
        "global_change_pct": global_rel_change,
        "global_velocity": global_velocity
    }

results_92_172 = analyze_interval(92, 172)
results_172_296 = analyze_interval(172, 296)

v1 = results_92_172["global_velocity"]
v2 = results_172_296["global_velocity"]
slowdown = (1.0 - (v2 / v1)) * 100.0 if v1 > 0 else 0.0

print("=" * 65)
print(f"        CONV2D WEIGHT VELOCITY DIAGNOSTIC ({'EMA' if USE_EMA else 'RAW'})")
print("=" * 65)
print(f"Interval 1: Epoch 92 -> 172 (Δ = 80 epochs)")
print(f"  - Total Weight Movement : {results_92_172['global_change_pct']:.2f}%")
print(f"  - Learning Velocity     : {v1:.4f}% weight shift per epoch")
print("-" * 65)
print(f"Interval 2: Epoch 172 -> 296 (Δ = 124 epochs)")
print(f"  - Total Weight Movement : {results_172_296['global_change_pct']:.2f}%")
print(f"  - Learning Velocity     : {v2:.4f}% weight shift per epoch")
print("=" * 65)
print(f"Velocity Deceleration Rate: {slowdown:.2f}% slowdown")
print("-" * 65)

if slowdown >= 80.0:
    print("VERDICT: [FULLY CONVERGED] Conv2D layers have stabilized (>80% velocity drop).")
elif slowdown >= 40.0:
    print("VERDICT: [SETTLING DOWN] Conv2D learning has significantly slowed down.")
else:
    print("VERDICT: [STILL LEARNING] Conv2D layers are actively moving at a steady pace.")
print("=" * 65 + "\n")

x = np.arange(len(dwconv_keys))
width = 0.35

fig, ax = plt.subplots(figsize=(14, 6))
fig.patch.set_facecolor('#121212')
ax.set_facecolor('#1e1e1e')

rects1 = ax.bar(x - width/2, results_92_172["velocities"], width, label='Epochs 92 -> 172', color='#4a90e2')
rects2 = ax.bar(x + width/2, results_172_296["velocities"], width, label='Epochs 172 -> 296', color='#e57373')

ax.set_ylabel('Velocity (% Weight Shift / Epoch)', color='white', fontsize=12)
ax.set_title(f'Conv2D Layer-by-Layer Learning Velocity ({ "EMA" if USE_EMA else "RAW" })', color='white', fontsize=14, pad=15)
ax.set_xticks(x)
ax.set_xticklabels(results_92_172["names"], rotation=45, ha='right', color='white', fontsize=9)
ax.tick_params(colors='white')
ax.grid(True, linestyle='--', alpha=0.2, color='white')
ax.legend(facecolor='#2a2a2a', edgecolor='none', labelcolor='white')

plt.tight_layout()
plt.savefig("conv2d_velocity_diagnostic.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
import math
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from samplers import cfg_velocity, get_schedule, get_1d_shifted_time

for module in model.modules():
    module._forward_hooks.clear()
    module._forward_pre_hooks.clear()

def plot_trajectory_curvature(model, initial_noise, text_embeds, text_mask, cfg=2.5, steps=30):
    device = initial_noise.device
    raw_timesteps = get_schedule(Config.validate_scheduler, steps, device)
    timesteps = get_1d_shifted_time(raw_timesteps, shift_val=2.5)
    
    x = initial_noise.clone().to(dtype=Config.dtype)
    velocities = []
    
    with torch.no_grad():
        for i in range(steps):
            t = timesteps[i].view(-1).to(dtype=Config.dtype)
            t_next = timesteps[i+1].view(-1).to(dtype=Config.dtype)
            dt = t_next - t
            
            with torch.amp.autocast("cuda", dtype=Config.dtype):
                v = cfg_velocity(model, x, t, text_embeds, cfg, text_mask=text_mask)
            
            velocities.append(v.flatten().float())
            x = x + v * dt
            x = x.to(dtype=Config.dtype)

    curvatures = []
    step_indices = list(range(1, steps))
    
    for i in range(len(velocities) - 1):
        v_curr = velocities[i]
        v_next = velocities[i+1]
        sim = F.cosine_similarity(v_curr.unsqueeze(0), v_next.unsqueeze(0)).item()
        curvatures.append(1.0 - sim)
        
    avg_curvature = np.mean(curvatures)
    
    fig, ax = plt.subplots(figsize=(10, 4.5))
    fig.patch.set_facecolor('#121212')
    ax.set_facecolor('#1e1e1e')
    
    ax.plot(step_indices, curvatures, color='#4fc3f7', linewidth=2.5, marker='o', markersize=5, label='Curvature (1 - CosSim)')
    ax.axhline(0.05, color='#ffb74d', linestyle='--', linewidth=1.5, label='Euler Threshold (0.05)')
    ax.axhline(avg_curvature, color='#e57373', linestyle=':', linewidth=1.5, label=f'Mean Curvature ({avg_curvature:.4f})')
    
    ax.set_xlabel('Sampling Step (i → i+1)', color='white', fontsize=11)
    ax.set_ylabel('Trajectory Curvature (1 - CosSim)', color='white', fontsize=11)
    ax.set_title(f'Flow Trajectory Curvature Across {steps} Steps (CFG = {cfg})', color='white', fontsize=13, pad=12)
    ax.tick_params(colors='white')
    ax.grid(True, linestyle='--', alpha=0.2, color='white')
    ax.legend(facecolor='#2a2a2a', edgecolor='none', labelcolor='white')
    
    plt.tight_layout()
    plt.savefig("trajectory_curvature_graph.png", dpi=150, bbox_inches='tight')
    plt.show()
    
    print("=" * 60)
    print(f"Average Trajectory Curvature: {avg_curvature:.6f}")
    if avg_curvature < 0.05:
        print("[VERDICT] Trajectory is extremely straight. Euler sampler is optimal!")
    else:
        print("[VERDICT] High trajectory curvature detected. Use RK4 or increase shift_val.")
    print("=" * 60)

plot_trajectory_curvature(model, initial_noise, combined_text_embeds, combined_mask)

In [ ]:
import os
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from latents import normalize_latents, load_repa_target

for module in model.modules():
    module._forward_hooks.clear()
    module._forward_pre_hooks.clear()

def probe_repa_alignment(model, target_filename="10003724.pt"):
    target_path = os.path.join(Config.cache_dir, target_filename)
    if not os.path.exists(target_path):
        raise FileNotFoundError(f"Target file not found at: {target_path}")
        
    data_pt = torch.load(target_path, map_location=DEVICE)
    clean_latent = normalize_latents(data_pt["latents"].unsqueeze(0).to(DEVICE, Config.dtype))
    repa_target = load_repa_target(data_pt, Config.dtype).unsqueeze(0).to(DEVICE)
    
    if "text_embeds_list" in data_pt:
        txt_emb = data_pt["text_embeds_list"][0].unsqueeze(0).to(DEVICE, Config.dtype)
        txt_msk = data_pt["attention_mask_list"][0].unsqueeze(0).to(DEVICE)
    else:
        txt_emb = data_pt["text_embeds"].unsqueeze(0).to(DEVICE, Config.dtype)
        txt_msk = data_pt["attention_mask"].unsqueeze(0).to(DEVICE)

    timesteps_to_test = [0.05, 0.20, 0.40, 0.60, 0.80, 0.95]
    cos_sims = []
    
    print("=" * 60)
    print(f"    REPA (DINOv3 Layer 8) ALIGNMENT PROBE ({target_filename})")
    print("=" * 60)
    
    with torch.no_grad():
        for t_val in timesteps_to_test:
            t = torch.tensor([t_val], device=DEVICE, dtype=Config.dtype)
            x_0 = torch.randn_like(clean_latent)
            x_t = (1.0 - t) * x_0 + t * clean_latent
            
            with torch.amp.autocast("cuda", dtype=Config.dtype):
                _, repa_pred = model(x_t, t, txt_emb, txt_msk, return_repa=True)
            
            sim = F.cosine_similarity(repa_pred.float(), repa_target.float(), dim=-1).mean().item()
            cos_sims.append(sim)
            print(f"Timestep t = {t_val:.2f} | DINOv3 Cosine Similarity: {sim:.4f}")
    print("=" * 60)
    
    fig, ax = plt.subplots(figsize=(8, 4))
    fig.patch.set_facecolor('#121212')
    ax.set_facecolor('#1e1e1e')
    
    ax.plot(timesteps_to_test, cos_sims, color='#81c784', linewidth=2.5, marker='s', markersize=6)
    ax.set_xlabel('Time t (0 = Pure Noise, 1 = Clean Image)', color='white', fontsize=11)
    ax.set_ylabel('DINOv3 Cosine Similarity', color='white', fontsize=11)
    ax.set_title('Layer 8 REPA Alignment Across Noise Trajectory', color='white', fontsize=13, pad=12)
    ax.tick_params(colors='white')
    ax.grid(True, linestyle='--', alpha=0.2, color='white')
    
    plt.tight_layout()
    plt.savefig("repa_alignment_graph.png", dpi=150, bbox_inches='tight')
    plt.show()

probe_repa_alignment(model, target_filename="10003724.pt")

In [ ]:
import math
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

for module in model.modules():
    module._forward_hooks.clear()
    module._forward_pre_hooks.clear()

def plot_text_image_attention(model, x, t, text_embeds, text_mask, block_idx=8):
    captured = {}
    
    def q_hook(module, input_args, output_tensor):
        captured['q'] = output_tensor.detach()

    def k_hook(module, input_args, output_tensor):
        captured['k'] = output_tensor.detach()

    target_block = model.blocks[block_idx]
    h1 = target_block.norm_q.register_forward_hook(q_hook)
    h2 = target_block.norm_k.register_forward_hook(k_hook)
    
    try:
        with torch.no_grad():
            with torch.amp.autocast("cuda", dtype=Config.dtype):
                _ = model(x.to(dtype=Config.dtype), t.to(dtype=Config.dtype), text_embeds.to(dtype=Config.dtype), text_mask)
            
        q = captured['q'].float()
        k = captured['k'].float()
        
        B, N, H, D = q.shape
        _, _, H_kv, _ = k.shape
        seq_len_text = text_embeds.shape[1]
        
        if H_kv != H:
            num_groups = H // H_kv
            k = k.repeat_interleave(num_groups, dim=2)
            
        q_img = q[:, seq_len_text:, :, :]
        k_txt = k[:, :seq_len_text, :, :]
        
        attn_logits = torch.einsum("b i h d, b j h d -> b h i j", q_img, k_txt) / math.sqrt(D)
        
        if text_mask is not None:
            mask_expanded = text_mask.unsqueeze(1).unsqueeze(2)
            attn_logits = attn_logits.masked_fill(~mask_expanded, float("-inf"))
            
        attn_probs = F.softmax(attn_logits, dim=-1)
        avg_attn_per_token = attn_probs.mean(dim=(0, 1, 2)).cpu().numpy()
        
        valid_lens = text_mask.sum().item() if text_mask is not None else seq_len_text
        
        token_indices = list(range(min(25, seq_len_text)))
        weights_to_plot = avg_attn_per_token[:len(token_indices)]
        
        fig, ax = plt.subplots(figsize=(11, 4.5))
        fig.patch.set_facecolor('#121212')
        ax.set_facecolor('#1e1e1e')
        
        colors = ['#ba68c8' if i < valid_lens else '#424242' for i in token_indices]
        ax.bar(token_indices, weights_to_plot, color=colors, width=0.6)
        
        ax.axvline(valid_lens - 0.5, color='#ff7043', linestyle='--', linewidth=1.5, label='End of Active Prompt')
        
        ax.set_xlabel('Text Token Index', color='white', fontsize=11)
        ax.set_ylabel('Mean Attention Weight', color='white', fontsize=11)
        ax.set_title(f'Cross-Attention Weight Allocated per Text Token (Block {block_idx})', color='white', fontsize=13, pad=12)
        ax.tick_params(colors='white')
        ax.grid(True, linestyle='--', alpha=0.2, color='white')
        ax.legend(facecolor='#2a2a2a', edgecolor='none', labelcolor='white')
        
        plt.tight_layout()
        plt.savefig("cross_attention_graph.png", dpi=150, bbox_inches='tight')
        plt.show()
        
    finally:
        h1.remove()
        h2.remove()

dummy_t = torch.tensor([0.5], device=DEVICE, dtype=Config.dtype)
plot_text_image_attention(model, initial_noise, dummy_t, text_embeds, text_mask, block_idx=8)

In [ ]:
import torch
import matplotlib.pyplot as plt
from samplers import get_schedule, get_1d_shifted_time, run_sampling_pipeline
from latents import decode_latents_to_image

for module in model.modules():
    module._forward_hooks.clear()
    module._forward_pre_hooks.clear()

def run_rescaled_cfg_sampling(model, initial_noise, steps, combined_text_embeds, cfg=7.0, rescaled_phi=0.7, text_mask=None):
    x = initial_noise.clone()
    raw_timesteps = get_schedule("uniform", steps, DEVICE)
    timesteps = get_1d_shifted_time(raw_timesteps, shift_val=2.5)
    
    for i in range(steps):
        t = timesteps[i].view(-1)
        t_next = timesteps[i+1].view(-1)
        dt = t_next - t
        
        with torch.no_grad():
            with torch.amp.autocast("cuda", dtype=Config.dtype):
                x_in = torch.cat([x, x], dim=0)
                t_in = torch.cat([t, t], dim=0)
                v_out = model(x_in, t_in, combined_text_embeds, text_mask=text_mask)
                v_uncond, v_cond = v_out.chunk(2, dim=0)
                
                v_cfg = v_uncond + cfg * (v_cond - v_uncond)
                
                std_cond = v_cond.std(dim=(1, 2, 3), keepdim=True)
                std_cfg = v_cfg.std(dim=(1, 2, 3), keepdim=True)
                v_rescaled = v_cfg * (std_cond / (std_cfg + 1e-6))
                v_final = rescaled_phi * v_rescaled + (1.0 - rescaled_phi) * v_cfg
                
                x = x + v_final * dt
                x = x.to(dtype=Config.dtype)
                
    return x

print("Generating high CFG samples (CFG = 7.0) to compare Standard vs Anti-Burn Rescaling...")

latents_rescaled = run_rescaled_cfg_sampling(
    model, initial_noise, steps=30, combined_text_embeds=combined_text_embeds, 
    cfg=7.0, rescaled_phi=0.7, text_mask=combined_mask
)
img_rescaled = decode_latents_to_image(vae, latents_rescaled, DEVICE)

with torch.no_grad():
    with torch.amp.autocast("cuda", dtype=Config.dtype):
        latents_std = run_sampling_pipeline(
            model=model, initial_noise=initial_noise.clone(), steps=30, 
            combined_text_embeds=combined_text_embeds, cfg=7.0, 
            text_mask=combined_mask, sampler_type="euler", 
            scheduler_type="uniform", shift_val=2.5
        )
img_std = decode_latents_to_image(vae, latents_std, DEVICE)

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(img_rescaled)
axes[0].set_title("CFG 7.0 + Variance Rescaling (Anti-Burn)", fontsize=12)
axes[0].axis("off")

axes[1].imshow(img_std)
axes[1].set_title("Standard CFG 7.0 (Saturated / Over-burned)", fontsize=12)
axes[1].axis("off")

plt.tight_layout()
plt.savefig("cfg_variance_rescaling_comparison.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from samplers import run_sampling_pipeline

for module in model.modules():
    module._forward_hooks.clear()
    module._forward_pre_hooks.clear()

def test_padding_invariance_graph(model, initial_noise, text_embeds, text_mask):
    valid_len = text_mask.sum().item()
    short_embeds = text_embeds[:, :valid_len, :]
    short_mask = text_mask[:, :valid_len]
    
    uncond_short_embeds = torch.zeros_like(short_embeds)
    uncond_short_mask = short_mask.clone()
    comb_short_embeds = torch.cat([uncond_short_embeds, short_embeds], dim=0)
    comb_short_mask = torch.cat([uncond_short_mask, short_mask], dim=0)
    
    pad_len = 64
    B, L, D = text_embeds.shape
    padded_embeds = torch.cat([text_embeds, torch.zeros(B, pad_len, D, device=DEVICE, dtype=Config.dtype)], dim=1)
    padded_mask = torch.cat([text_mask, torch.zeros(B, pad_len, device=DEVICE, dtype=torch.bool)], dim=1)
    
    uncond_padded_embeds = torch.zeros_like(padded_embeds)
    uncond_padded_mask = padded_mask.clone()
    comb_padded_embeds = torch.cat([uncond_padded_embeds, padded_embeds], dim=0)
    comb_padded_mask = torch.cat([uncond_padded_mask, padded_mask], dim=0)
    
    with torch.no_grad():
        with torch.amp.autocast("cuda", dtype=Config.dtype):
            latents_short = run_sampling_pipeline(
                model, initial_noise.clone().to(dtype=Config.dtype), steps=20, 
                combined_text_embeds=comb_short_embeds, cfg=2.5, 
                text_mask=comb_short_mask, shift_val=2.5
            )
            latents_padded = run_sampling_pipeline(
                model, initial_noise.clone().to(dtype=Config.dtype), steps=20, 
                combined_text_embeds=comb_padded_embeds, cfg=2.5, 
                text_mask=comb_padded_mask, shift_val=2.5
            )

    diff_map = (latents_short[0].float() - latents_padded[0].float()).abs().mean(dim=0).cpu().numpy()
    flat_diffs = (latents_short.float() - latents_padded.float()).abs().flatten().cpu().numpy()
    max_diff = flat_diffs.max()

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    fig.patch.set_facecolor('#121212')
    
    axes[0].set_facecolor('#1e1e1e')
    im0 = axes[0].imshow(diff_map, cmap='viridis')
    axes[0].set_title(f"Spatial Difference Map (Max Diff: {max_diff:.2e})", color='white', fontsize=12)
    axes[0].axis('off')
    cbar0 = fig.colorbar(im0, ax=axes[0], shrink=0.8)
    cbar0.ax.yaxis.set_tick_params(color='white', labelcolor='white')

    axes[1].set_facecolor('#1e1e1e')
    axes[1].hist(flat_diffs, bins=50, color='#4fc3f7', edgecolor='#121212')
    axes[1].axvline(1e-3, color='#e57373', linestyle='--', label='Tolerance Limit (1e-3)')
    axes[1].set_xlabel('Absolute Pixel Error', color='white', fontsize=11)
    axes[1].set_ylabel('Count', color='white', fontsize=11)
    axes[1].set_title('Padding Invariance Error Distribution', color='white', fontsize=12)
    axes[1].tick_params(colors='white')
    axes[1].grid(True, linestyle='--', alpha=0.2, color='white')
    axes[1].legend(facecolor='#2a2a2a', edgecolor='none', labelcolor='white')

    plt.tight_layout()
    plt.savefig("padding_invariance_graph.png", dpi=150, bbox_inches='tight')
    plt.show()

test_padding_invariance_graph(model, initial_noise, text_embeds, text_mask)

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

def plot_2d_frequency_spectrum(latents):
    latent_spatial = latents[0].mean(dim=0).float().cpu().numpy()
    
    fft_2d = np.fft.fft2(latent_spatial)
    fft_shifted = np.fft.fftshift(fft_2d)
    magnitude_spectrum = np.log(np.abs(fft_shifted) + 1e-6)
    
    fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
    fig.patch.set_facecolor('#121212')
    
    axes[0].set_facecolor('#1e1e1e')
    im0 = axes[0].imshow(latent_spatial, cmap='magma')
    axes[0].set_title('Spatial Latent Map (Channel Mean)', color='white', fontsize=12)
    axes[0].axis('off')
    
    axes[1].set_facecolor('#1e1e1e')
    im1 = axes[1].imshow(magnitude_spectrum, cmap='inferno')
    axes[1].set_title('2D Power Spectrum (FFT)', color='white', fontsize=12)
    axes[1].axis('off')
    
    plt.tight_layout()
    plt.savefig("frequency_spectrum_fft.png", dpi=150, bbox_inches='tight')
    plt.show()
    print("[DIAGNOSTIC] Check FFT map: High-frequency energy should be centrally concentrated without bright outer grid spots.")

with torch.no_grad():
    with torch.amp.autocast("cuda", dtype=Config.dtype):
        sample_latents = run_sampling_pipeline(model, initial_noise, steps=25, combined_text_embeds=combined_text_embeds, cfg=2.5, text_mask=combined_mask)

plot_2d_frequency_spectrum(sample_latents)

In [ ]:
import torch
import matplotlib.pyplot as plt
from samplers import run_sampling_pipeline
from latents import decode_latents_to_image

for module in model.modules():
    module._forward_hooks.clear()
    module._forward_pre_hooks.clear()

def test_aspect_ratio_graph(model, text_embeds, text_mask):
    h_custom, w_custom = 384, 640
    
    uncond_emb = torch.zeros_like(text_embeds)
    uncond_msk = text_mask.clone()
    comb_emb = torch.cat([uncond_emb, text_embeds], dim=0)
    comb_msk = torch.cat([uncond_msk, text_mask], dim=0)
    
    gen = torch.Generator(device=DEVICE).manual_seed(42)
    custom_noise = torch.randn(
        1, Config.in_channels, 
        h_custom // Config.vae_downsample_factor, 
        w_custom // Config.vae_downsample_factor, 
        generator=gen, device=DEVICE, dtype=Config.dtype
    )
    
    with torch.no_grad():
        with torch.amp.autocast("cuda", dtype=Config.dtype):
            latents_custom = run_sampling_pipeline(
                model, custom_noise, steps=30, 
                combined_text_embeds=comb_emb, cfg=2.5, 
                text_mask=comb_msk, shift_val=2.5
            )
            
    img_custom = decode_latents_to_image(vae, latents_custom, DEVICE)
    
    lat_spatial = latents_custom[0].float().cpu()
    row_var = lat_spatial.var(dim=(0, 2)).numpy()
    col_var = lat_spatial.var(dim=(0, 1)).numpy()

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.patch.set_facecolor('#121212')
    
    axes[0].imshow(img_custom)
    axes[0].set_title(f"Generated Sample ({w_custom}x{h_custom})", color='white', fontsize=12)
    axes[0].axis('off')

    axes[1].set_facecolor('#1e1e1e')
    axes[1].plot(np.linspace(0, 1, len(row_var)), row_var, color='#81c784', linewidth=2, label='Vertical Axis Profile (Height)')
    axes[1].plot(np.linspace(0, 1, len(col_var)), col_var, color='#64b5f6', linewidth=2, label='Horizontal Axis Profile (Width)')
    axes[1].set_xlabel('Normalized Spatial Position (0 → 1)', color='white', fontsize=11)
    axes[1].set_ylabel('Latent Variance', color='white', fontsize=11)
    axes[1].set_title('3D RoPE Axis Activation Profiles', color='white', fontsize=12)
    axes[1].tick_params(colors='white')
    axes[1].grid(True, linestyle='--', alpha=0.2, color='white')
    axes[1].legend(facecolor='#2a2a2a', edgecolor='none', labelcolor='white')

    plt.tight_layout()
    plt.savefig("aspect_ratio_graph.png", dpi=150, bbox_inches='tight')
    plt.show()

test_aspect_ratio_graph(model, text_embeds, text_mask)

In [ ]:
import math
import numpy as np
import torch
import matplotlib.pyplot as plt
from samplers import get_schedule, get_1d_shifted_time

for module in model.modules():
    module._forward_hooks.clear()
    module._forward_pre_hooks.clear()

def plot_guidance_magnitude_trajectory(model, initial_noise, combined_text_embeds, combined_mask, steps=30):
    device = initial_noise.device
    raw_timesteps = get_schedule(Config.validate_scheduler, steps, device)
    timesteps = get_1d_shifted_time(raw_timesteps, shift_val=2.5)
    
    x = initial_noise.clone().to(dtype=Config.dtype)
    guidance_norms = []
    uncond_norms = []
    cond_norms = []
    
    with torch.no_grad():
        for i in range(steps):
            t = timesteps[i].view(-1).to(dtype=Config.dtype)
            t_next = timesteps[i+1].view(-1).to(dtype=Config.dtype)
            dt = t_next - t
            
            with torch.amp.autocast("cuda", dtype=Config.dtype):
                x_in = torch.cat([x, x], dim=0)
                t_in = torch.cat([t, t], dim=0)
                v_out = model(x_in, t_in, combined_text_embeds, text_mask=combined_mask)
                v_uncond, v_cond = v_out.chunk(2, dim=0)
                
                v_diff = v_cond - v_uncond
                g_norm = v_diff.norm(p=2, dim=(1, 2, 3)).mean().item()
                u_norm = v_uncond.norm(p=2, dim=(1, 2, 3)).mean().item()
                c_norm = v_cond.norm(p=2, dim=(1, 2, 3)).mean().item()
                
                guidance_norms.append(g_norm)
                uncond_norms.append(u_norm)
                cond_norms.append(c_norm)
                
                v_final = v_uncond + 2.5 * v_diff
                x = x + v_final * dt
                x = x.to(dtype=Config.dtype)

    step_indices = list(range(1, steps + 1))
    fig, ax = plt.subplots(figsize=(10, 4.5))
    fig.patch.set_facecolor('#121212')
    ax.set_facecolor('#1e1e1e')
    
    ax.plot(step_indices, guidance_norms, color='#ff7043', linewidth=2.5, marker='o', markersize=5, label='Text Steering Norm ||v_cond - v_uncond||')
    ax.plot(step_indices, cond_norms, color='#4fc3f7', linewidth=1.8, linestyle='--', label='Cond Velocity Norm ||v_cond||')
    ax.plot(step_indices, uncond_norms, color='#81c784', linewidth=1.8, linestyle=':', label='Uncond Velocity Norm ||v_uncond||')
    
    ax.set_xlabel('Sampling Step (1 → 30)', color='white', fontsize=11)
    ax.set_ylabel('L2 Vector Norm', color='white', fontsize=11)
    ax.set_title('Text Guidance Magnitude Across Sampling Trajectory', color='white', fontsize=13, pad=12)
    ax.tick_params(colors='white')
    ax.grid(True, linestyle='--', alpha=0.2, color='white')
    ax.legend(facecolor='#2a2a2a', edgecolor='none', labelcolor='white')
    
    plt.tight_layout()
    plt.savefig("guidance_magnitude_graph.png", dpi=150, bbox_inches='tight')
    plt.show()

plot_guidance_magnitude_trajectory(model, initial_noise, combined_text_embeds, combined_mask)

In [ ]:
import torch
import matplotlib.pyplot as plt

for module in model.modules():
    module._forward_hooks.clear()
    module._forward_pre_hooks.clear()

def plot_layer_activation_norms(model, x, t, text_embeds, text_mask):
    layer_norms = []
    layer_names = []
    
    hooks = []
    captured_outputs = {}
    
    def get_hook(idx):
        def hook_fn(module, input_args, output_tensor):
            captured_outputs[idx] = output_tensor.detach()
        return hook_fn

    for idx, block in enumerate(model.blocks):
        h = block.register_forward_hook(get_hook(idx))
        hooks.append(h)
        layer_names.append(f"block.{idx}")
        
    try:
        with torch.no_grad():
            with torch.amp.autocast("cuda", dtype=Config.dtype):
                _ = model(x.to(dtype=Config.dtype), t.to(dtype=Config.dtype), text_embeds.to(dtype=Config.dtype), text_mask)
                
        for idx in range(len(model.blocks)):
            out_tensor = captured_outputs[idx].float()
            norm_val = out_tensor.norm(p=2, dim=-1).mean().item()
            layer_norms.append(norm_val)
            
        fig, ax = plt.subplots(figsize=(11, 4.5))
        fig.patch.set_facecolor('#121212')
        ax.set_facecolor('#1e1e1e')
        
        x_indices = list(range(len(layer_norms)))
        ax.plot(x_indices, layer_norms, color='#ba68c8', linewidth=2.5, marker='s', markersize=6, label='Residual Norm ||h_l||')
        
        ax.set_xlabel('Transformer Block Index (0 → 15)', color='white', fontsize=11)
        ax.set_ylabel('Mean Activation Norm', color='white', fontsize=11)
        ax.set_title('Residual Stream Feature Magnitude Across Network Depth', color='white', fontsize=13, pad=12)
        ax.set_xticks(x_indices)
        ax.set_xticklabels(layer_names, rotation=45, ha='right', color='white', fontsize=9)
        ax.tick_params(colors='white')
        ax.grid(True, linestyle='--', alpha=0.2, color='white')
        ax.legend(facecolor='#2a2a2a', edgecolor='none', labelcolor='white')
        
        plt.tight_layout()
        plt.savefig("residual_stream_norms_graph.png", dpi=150, bbox_inches='tight')
        plt.show()
        
    finally:
        for h in hooks:
            h.remove()

dummy_t = torch.tensor([0.5], device=DEVICE, dtype=Config.dtype)
plot_layer_activation_norms(model, initial_noise, dummy_t, text_embeds, text_mask)

In [ ]:
import math
import numpy as np
import torch
import matplotlib.pyplot as plt
from samplers import cfg_velocity, get_schedule, get_1d_shifted_time

for module in model.modules():
    module._forward_hooks.clear()
    module._forward_pre_hooks.clear()

def plot_latent_variance_trajectory(model, initial_noise, text_embeds, text_mask, cfg=2.5, steps=30):
    device = initial_noise.device
    raw_timesteps = get_schedule(Config.validate_scheduler, steps, device)
    timesteps = get_1d_shifted_time(raw_timesteps, shift_val=2.5)
    
    x = initial_noise.clone().to(dtype=Config.dtype)
    means = []
    stds = []
    
    with torch.no_grad():
        for i in range(steps):
            t = timesteps[i].view(-1).to(dtype=Config.dtype)
            t_next = timesteps[i+1].view(-1).to(dtype=Config.dtype)
            dt = t_next - t
            
            means.append(x.float().mean().item())
            stds.append(x.float().std().item())
            
            with torch.amp.autocast("cuda", dtype=Config.dtype):
                v = cfg_velocity(model, x, t, text_embeds, cfg, text_mask=text_mask)
            
            x = x + v * dt
            x = x.to(dtype=Config.dtype)
            
    means.append(x.float().mean().item())
    stds.append(x.float().std().item())

    step_indices = list(range(steps + 1))
    fig, ax = plt.subplots(figsize=(10, 4.5))
    fig.patch.set_facecolor('#121212')
    ax.set_facecolor('#1e1e1e')
    
    ax.plot(step_indices, stds, color='#64b5f6', linewidth=2.5, label='Latent Std Dev σ(x_t)')
    ax.plot(step_indices, means, color='#e57373', linewidth=2.0, linestyle='--', label='Latent Mean μ(x_t)')
    ax.axhline(1.0, color='#81c784', linestyle=':', label='Target Unit Variance (1.0)')
    
    ax.set_xlabel('Sampling Step (0 → 30)', color='white', fontsize=11)
    ax.set_ylabel('Value', color='white', fontsize=11)
    ax.set_title('Latent Distribution Stability During Sampling', color='white', fontsize=13, pad=12)
    ax.tick_params(colors='white')
    ax.grid(True, linestyle='--', alpha=0.2, color='white')
    ax.legend(facecolor='#2a2a2a', edgecolor='none', labelcolor='white')
    
    plt.tight_layout()
    plt.savefig("latent_variance_trajectory_graph.png", dpi=150, bbox_inches='tight')
    plt.show()

plot_latent_variance_trajectory(model, initial_noise, combined_text_embeds, combined_mask)